# MVRV Z-Score + LTH-SOPR Exit Signal

**Hypothesis:** Combining MVRV Z-Score (relative valuation) with LTH-SOPR (smart money distribution) provides better exit timing.

**Why MVRV Z might be better than raw MVRV:**
- Raw MVRV > 2.5 is an absolute threshold
- MVRV Z > 2.0 means "2 std devs above historical mean"
- Z-score adapts to changing market structure
- More statistically meaningful

**The Combination:**
- MVRV Z > X = Market is statistically overvalued
- LTH-SOPR > Y = Smart money is actually selling
- Together = "Expensive AND distribution happening"

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from numba import njit
import warnings
warnings.filterwarnings('ignore')

print("MVRV Z-Score + LTH-SOPR Exit Analysis 📊")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
sopr_lth = pd.read_parquet(DATA_DIR / "sopr_lth.parquet").rename(columns={"value": "sopr_lth"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
mvrv_z = pd.read_parquet(DATA_DIR / "mvrv_z.parquet").rename(columns={"value": "mvrv_z"}).set_index("time")

df = price.join(sopr_sth, how='inner').join(sopr_lth, how='inner').join(mvrv, how='inner').join(mvrv_z, how='inner')
df = df.sort_index()
df = df[df.index >= '2019-01-01'].dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")
print(f"\nMVRV Z-Score stats:")
print(f"  Min: {df['mvrv_z'].min():.2f}")
print(f"  Max: {df['mvrv_z'].max():.2f}")
print(f"  Mean: {df['mvrv_z'].mean():.2f}")
print(f"  Std: {df['mvrv_z'].std():.2f}")
print(f"  % > 2.0: {(df['mvrv_z'] > 2).mean()*100:.1f}%")
print(f"  % > 3.0: {(df['mvrv_z'] > 3).mean()*100:.1f}%")

In [ ]:
# Forward returns
df['fwd_30d'] = df['price'].shift(-30) / df['price'] - 1
df['fwd_90d'] = df['price'].shift(-90) / df['price'] - 1

print("Data ready for analysis")

---
## 1. Visualize MVRV vs MVRV Z-Score

In [ ]:
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.4, 0.2, 0.2, 0.2],
    subplot_titles=('BTC Price (Log)', 'MVRV (Raw)', 'MVRV Z-Score', 'LTH-SOPR')
)

fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price', line=dict(color='orange')), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['mvrv'], name='MVRV', line=dict(color='green')), row=2, col=1)
fig.add_hline(y=2.0, line_dash='dash', line_color='orange', row=2, col=1)
fig.add_hline(y=2.5, line_dash='dash', line_color='red', row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['mvrv_z'], name='MVRV Z', line=dict(color='blue')), row=3, col=1)
fig.add_hline(y=2.0, line_dash='dash', line_color='orange', row=3, col=1)
fig.add_hline(y=3.0, line_dash='dash', line_color='red', row=3, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=3, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['sopr_lth'], name='LTH-SOPR', line=dict(color='purple')), row=4, col=1)
fig.add_hline(y=1.5, line_dash='dash', line_color='orange', row=4, col=1)
fig.add_hline(y=2.0, line_dash='dash', line_color='red', row=4, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=1000, title='MVRV vs MVRV Z-Score vs LTH-SOPR', showlegend=False)
fig.show()

---
## 2. At Major Tops: MVRV vs MVRV Z

In [ ]:
major_tops = [
    ('2021-04-14', 'April 2021 ATH'),
    ('2021-11-10', 'Nov 2021 ATH'),
    ('2024-03-14', 'March 2024 ATH'),
    ('2024-12-17', 'Dec 2024 ATH'),
    ('2021-05-10', 'Pre-China Crash'),
    ('2022-03-28', 'Bear Rally 1'),
]

print("METRICS AT MAJOR TOPS")
print("="*120)
print(f"{'Date':<12} {'Event':<20} {'Price':>10} {'MVRV':>8} {'MVRV Z':>8} {'LTH-SOPR':>10} {'30d Fwd':>10}")
print("-"*100)

top_data = []
for date_str, name in major_tops:
    try:
        target = pd.Timestamp(date_str, tz='UTC')
        idx = df.index.get_indexer([target], method='nearest')[0]
        row = df.iloc[idx]
        actual_date = df.index[idx]
        
        top_data.append({
            'date': actual_date,
            'name': name,
            'mvrv': row['mvrv'],
            'mvrv_z': row['mvrv_z'],
            'lth_sopr': row['sopr_lth'],
            'fwd_30d': row['fwd_30d'],
        })
        
        fwd = f"{row['fwd_30d']*100:+.0f}%" if pd.notna(row['fwd_30d']) else 'N/A'
        print(f"{actual_date.strftime('%Y-%m-%d'):<12} {name:<20} ${row['price']:>9,.0f} {row['mvrv']:>8.2f} {row['mvrv_z']:>8.2f} {row['sopr_lth']:>10.2f} {fwd:>10}")
    except Exception as e:
        print(f"Error: {e}")

tops_df = pd.DataFrame(top_data)

print(f"\nSummary at Tops:")
print(f"  Avg MVRV: {tops_df['mvrv'].mean():.2f}")
print(f"  Avg MVRV Z: {tops_df['mvrv_z'].mean():.2f}")
print(f"  Avg LTH-SOPR: {tops_df['lth_sopr'].mean():.2f}")
print(f"  % with MVRV > 2.0: {(tops_df['mvrv'] > 2.0).mean()*100:.0f}%")
print(f"  % with MVRV Z > 2.0: {(tops_df['mvrv_z'] > 2.0).mean()*100:.0f}%")
print(f"  % with MVRV Z > 3.0: {(tops_df['mvrv_z'] > 3.0).mean()*100:.0f}%")

---
## 3. Forward Returns by Signal

In [ ]:
print("FORWARD RETURNS BY EXIT SIGNAL")
print("="*110)
print(f"{'Signal':<45} {'Avg 30d':>10} {'Avg 90d':>10} {'Days':>8} {'% Time':>8}")
print("-"*110)

exit_signals = [
    ('All days', df['mvrv'] > 0),
    
    # Raw MVRV (baseline)
    ('MVRV > 2.0', df['mvrv'] > 2.0),
    ('MVRV > 2.5', df['mvrv'] > 2.5),
    ('MVRV > 3.0', df['mvrv'] > 3.0),
    
    # MVRV Z-Score
    ('MVRV Z > 1.5', df['mvrv_z'] > 1.5),
    ('MVRV Z > 2.0', df['mvrv_z'] > 2.0),
    ('MVRV Z > 2.5', df['mvrv_z'] > 2.5),
    ('MVRV Z > 3.0', df['mvrv_z'] > 3.0),
    ('MVRV Z > 4.0', df['mvrv_z'] > 4.0),
    
    # LTH-SOPR alone
    ('LTH-SOPR > 1.5', df['sopr_lth'] > 1.5),
    ('LTH-SOPR > 2.0', df['sopr_lth'] > 2.0),
    ('LTH-SOPR > 3.0', df['sopr_lth'] > 3.0),
    
    # MVRV + LTH (from previous test)
    ('MVRV>2.5 + LTH>1.5', (df['mvrv'] > 2.5) & (df['sopr_lth'] > 1.5)),
    
    # MVRV Z + LTH combinations
    ('MVRV Z>1.5 + LTH>1.5', (df['mvrv_z'] > 1.5) & (df['sopr_lth'] > 1.5)),
    ('MVRV Z>2.0 + LTH>1.5', (df['mvrv_z'] > 2.0) & (df['sopr_lth'] > 1.5)),
    ('MVRV Z>2.0 + LTH>2.0', (df['mvrv_z'] > 2.0) & (df['sopr_lth'] > 2.0)),
    ('MVRV Z>2.5 + LTH>1.5', (df['mvrv_z'] > 2.5) & (df['sopr_lth'] > 1.5)),
    ('MVRV Z>2.5 + LTH>2.0', (df['mvrv_z'] > 2.5) & (df['sopr_lth'] > 2.0)),
    ('MVRV Z>3.0 + LTH>1.5', (df['mvrv_z'] > 3.0) & (df['sopr_lth'] > 1.5)),
    ('MVRV Z>3.0 + LTH>2.0', (df['mvrv_z'] > 3.0) & (df['sopr_lth'] > 2.0)),
    ('MVRV Z>3.0 + LTH>3.0', (df['mvrv_z'] > 3.0) & (df['sopr_lth'] > 3.0)),
]

results = []
for name, cond in exit_signals:
    subset = df[cond]
    if len(subset) > 5:
        avg_30 = subset['fwd_30d'].mean() * 100
        avg_90 = subset['fwd_90d'].mean() * 100
        pct = len(subset) / len(df) * 100
        results.append({'name': name, 'avg_30': avg_30, 'avg_90': avg_90, 'days': len(subset), 'pct': pct})
        marker = '⭐' if avg_90 < 0 else ''
        print(f"{name:<45} {avg_30:>+9.1f}% {avg_90:>+9.1f}% {len(subset):>8} {pct:>7.1f}% {marker}")

In [ ]:
# Rank by most negative forward returns
print("\n" + "="*70)
print("BEST EXIT SIGNALS (Most Negative 90d Forward Returns)")
print("="*70)

sorted_results = sorted(results, key=lambda x: x['avg_90'])
for i, r in enumerate(sorted_results[:15]):
    quality = '⭐' if r['avg_90'] < 0 else ''
    print(f"{i+1:>2}. {r['name']:<45} {r['avg_90']:>+8.1f}% ({r['pct']:.1f}% of time) {quality}")

---
## 4. Backtest Exit Strategies

In [ ]:
@njit
def exit_simple_trail(price_arr, lth_arr, mvrv_arr, mvrv_z_arr, entry_idx, trail_pct=0.30):
    """Baseline: Simple trailing stop"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price > peak:
            peak = price
        if price <= peak * (1 - trail_pct):
            return j, price, 'trail'
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_mvrv_lth(price_arr, lth_arr, mvrv_arr, mvrv_z_arr, entry_idx,
                  mvrv_context=2.5, lth_exit=1.5,
                  trail_before=0.30, trail_after=0.15):
    """MVRV (raw) + LTH-SOPR trigger"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    triggered = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        lth = lth_arr[j]
        mvrv = mvrv_arr[j]
        
        if price > peak:
            peak = price
        
        if not triggered:
            if mvrv > mvrv_context and lth > lth_exit:
                triggered = True
        
        trail = trail_after if triggered else trail_before
        
        if price <= peak * (1 - trail):
            reason = 'mvrv_lth' if triggered else 'trail'
            return j, price, reason
    
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_mvrvz_lth(price_arr, lth_arr, mvrv_arr, mvrv_z_arr, entry_idx,
                   mvrv_z_context=2.0, lth_exit=1.5,
                   trail_before=0.30, trail_after=0.15):
    """MVRV Z-Score + LTH-SOPR trigger"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    triggered = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        lth = lth_arr[j]
        mvrv_z = mvrv_z_arr[j]
        
        if price > peak:
            peak = price
        
        if not triggered:
            if mvrv_z > mvrv_z_context and lth > lth_exit:
                triggered = True
        
        trail = trail_after if triggered else trail_before
        
        if price <= peak * (1 - trail):
            reason = 'mvrvz_lth' if triggered else 'trail'
            return j, price, reason
    
    return len(price_arr) - 1, price_arr[-1], 'hold'

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, **kwargs):
    price_arr = df['price'].values
    lth_arr = df['sopr_lth'].values
    mvrv_arr = df['mvrv'].values
    mvrv_z_arr = df['mvrv_z'].values
    dates = df.index
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        exit_idx, exit_price, exit_reason = exit_func(price_arr, lth_arr, mvrv_arr, mvrv_z_arr, entry_idx, **kwargs)
        
        entry_price = price_arr[entry_idx]
        net_return = (exit_price / entry_price) - 1 - 0.002
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'exit_reason': exit_reason,
            'exit_lth': lth_arr[exit_idx],
            'exit_mvrv': mvrv_arr[exit_idx],
            'exit_mvrv_z': mvrv_z_arr[exit_idx],
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    return trades_df


def calc_metrics(trades, initial_capital=100000):
    if len(trades) == 0:
        return None
    final = trades['equity'].iloc[-1]
    total_ret = (final / initial_capital) - 1
    years = (trades['exit_date'].iloc[-1] - trades['entry_date'].iloc[0]).days / 365.25
    win_rate = (trades['net_return'] > 0).mean()
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(trades)/years) if returns.std() > 0 and years > 0 else 0
    
    equity = [initial_capital] + list(trades['equity'])
    peak, max_dd = equity[0], 0
    for eq in equity:
        if eq > peak: peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd: max_dd = dd
    
    return {
        'total_return': total_ret,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'win_rate': win_rate,
        'n_trades': len(trades),
        'avg_hold': trades['days_held'].mean()
    }

In [ ]:
# Entry: STH-SOPR < 1 (STRAT-003)
entry_cond = df['sopr_sth'] < 1
entries = entry_cond & ~entry_cond.shift(1).fillna(False)

print("STRAT-003: MVRV Z + LTH-SOPR EXIT COMPARISON")
print("Entry: STH-SOPR < 1")
print("="*120)

strategies = [
    # Baselines
    ('Simple 30% Trail', exit_simple_trail, {'trail_pct': 0.30}),
    ('Simple 15% Trail', exit_simple_trail, {'trail_pct': 0.15}),
    
    # MVRV + LTH (previous best)
    ('MVRV>2.5 + LTH>1.5 → 30/15', exit_mvrv_lth,
     {'mvrv_context': 2.5, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    
    # MVRV Z + LTH combinations
    ('MVRV Z>1.5 + LTH>1.5 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 1.5, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('MVRV Z>2.0 + LTH>1.5 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 2.0, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('MVRV Z>2.0 + LTH>2.0 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 2.0, 'lth_exit': 2.0, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('MVRV Z>2.5 + LTH>1.5 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 2.5, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('MVRV Z>2.5 + LTH>2.0 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 2.5, 'lth_exit': 2.0, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('MVRV Z>3.0 + LTH>1.5 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 3.0, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('MVRV Z>3.0 + LTH>2.0 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 3.0, 'lth_exit': 2.0, 'trail_before': 0.30, 'trail_after': 0.15}),
    
    # Try tighter after-trigger trails
    ('MVRV Z>2.0 + LTH>1.5 → 30/10', exit_mvrvz_lth,
     {'mvrv_z_context': 2.0, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.10}),
    ('MVRV Z>2.5 + LTH>1.5 → 30/10', exit_mvrvz_lth,
     {'mvrv_z_context': 2.5, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.10}),
]

print(f"{'Strategy':<40} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'Trades':>7} {'AvgHold':>8}")
print("-"*100)

all_results = []
for name, func, kwargs in strategies:
    trades = run_backtest(df, entries, func, **kwargs)
    m = calc_metrics(trades)
    if m:
        print(f"{name:<40} {m['total_return']*100:>+9.0f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {m['n_trades']:>7} {m['avg_hold']:>7.0f}d")
        all_results.append({'name': name, 'metrics': m, 'trades': trades})

In [ ]:
# Find winners
best_simple = max([r for r in all_results if 'Simple' in r['name']], key=lambda x: x['metrics']['total_return'])
best_mvrv = [r for r in all_results if 'MVRV>' in r['name'] and 'Z' not in r['name']]
best_mvrv = max(best_mvrv, key=lambda x: x['metrics']['total_return']) if best_mvrv else None
best_mvrvz = max([r for r in all_results if 'MVRV Z' in r['name']], key=lambda x: x['metrics']['total_return'])

print("\n" + "="*80)
print("COMPARISON: Simple vs MVRV vs MVRV Z")
print("="*80)

print(f"\n{'Strategy':<45} {'Return':>15}")
print("-"*60)
print(f"{best_simple['name']:<45} {best_simple['metrics']['total_return']*100:>+14.0f}%")
if best_mvrv:
    print(f"{best_mvrv['name']:<45} {best_mvrv['metrics']['total_return']*100:>+14.0f}%")
print(f"{best_mvrvz['name']:<45} {best_mvrvz['metrics']['total_return']*100:>+14.0f}%")

overall_best = max(all_results, key=lambda x: x['metrics']['total_return'])
print(f"\n🏆 OVERALL WINNER: {overall_best['name']}")
print(f"   Return: {overall_best['metrics']['total_return']*100:+,.0f}%")

In [ ]:
# Trade log for best MVRV Z strategy
print("\n" + "="*110)
print(f"TRADE LOG: {best_mvrvz['name']}")
print("="*110)

t = best_mvrvz['trades']
print(f"\n{'Entry':<12} {'Exit':<12} {'Days':>6} {'Return':>10} {'Reason':<12} {'MVRV Z':>8} {'LTH':>8}")
print("-"*85)

for _, row in t.iterrows():
    print(f"{row['entry_date'].strftime('%Y-%m-%d'):<12} {row['exit_date'].strftime('%Y-%m-%d'):<12} {row['days_held']:>6} {row['net_return']*100:>+9.0f}% {row['exit_reason']:<12} {row['exit_mvrv_z']:>8.2f} {row['exit_lth']:>8.2f}")

---
## 5. Test with STRAT-002 Entry (Long-term)

In [ ]:
# Load realized loss for full entry
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")
df = df.join(realized_loss, how='inner')
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
df = df.join(sopr, how='inner')

# Full STRAT-002 entry
entry_cond_full = (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['rl_zscore'] > 0.5)
entries_full = entry_cond_full & ~entry_cond_full.shift(1).fillna(False)

print("\nSTRAT-002 ENTRY + MVRV Z EXIT")
print("Entry: SOPR < 1 AND STH-SOPR < 1 AND RL Z > 0.5")
print("="*100)

strat2_tests = [
    ('Simple 30% Trail (v5)', exit_simple_trail, {'trail_pct': 0.30}),
    ('MVRV>2.5 + LTH>1.5 → 30/15', exit_mvrv_lth,
     {'mvrv_context': 2.5, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('MVRV Z>2.0 + LTH>1.5 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 2.0, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('MVRV Z>2.5 + LTH>1.5 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 2.5, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('MVRV Z>3.0 + LTH>1.5 → 30/15', exit_mvrvz_lth,
     {'mvrv_z_context': 3.0, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
]

print(f"{'Strategy':<40} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'Trades':>7}")
print("-"*90)

for name, func, kwargs in strat2_tests:
    trades = run_backtest(df, entries_full, func, **kwargs)
    m = calc_metrics(trades)
    if m:
        print(f"{name:<40} {m['total_return']*100:>+9.0f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {m['n_trades']:>7}")

---
## 6. Summary

In [ ]:
print("\n" + "="*70)
print("MVRV Z-SCORE + LTH-SOPR EXIT SUMMARY")
print("="*70)

print(f"""
📊 MVRV Z-SCORE vs RAW MVRV:
   • MVRV Z > 2.0 = Market is 2 std devs above mean (statistically expensive)
   • MVRV > 2.5 = Market is expensive (absolute threshold)
   • Z-score adapts to changing market structure
   • At tops: Avg MVRV Z = {tops_df['mvrv_z'].mean():.1f}

📈 STRAT-003 (Short-term) RESULTS:
   Simple Trail: {best_simple['metrics']['total_return']*100:+,.0f}%
   Best MVRV Z Exit: {best_mvrvz['metrics']['total_return']*100:+,.0f}%
   Winner: {'MVRV Z ✅' if best_mvrvz['metrics']['total_return'] > best_simple['metrics']['total_return'] else 'Simple ✅'}

💡 KEY INSIGHT:
   • MVRV Z provides statistical context (std devs from mean)
   • Raw MVRV provides absolute context (market cap vs realized cap)
   • LTH-SOPR provides action (smart money selling)
   • Best combo: Context (MVRV or MVRV Z) + Action (LTH-SOPR)
""")